# LangGraph com Checkpointer de Memória AgentCore (Memória de curto prazo)

## Introdução
Este notebook demonstra como integrar as capacidades de Memória do Amazon Bedrock AgentCore com LangGraph usando o checkpointer **AgentCoreMemorySaver**. Vamos focar na persistência de **memória de curto prazo** entre turnos de conversa - permitindo que um agente mantenha contexto contínuo e construa sobre cálculos anteriores através de checkpointing automático de estado.

## Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Curto Prazo Conversacional                                                       |
| Caso de uso do agente | Cálculos Matemáticos Multi-Etapa                                              |
| Framework Agêntico  | Langgraph                                                                        |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | AgentCoreMemorySaver Checkpointer, Ferramentas Matemáticas                   |
| Complexidade do exemplo | Intermediária                                                                |

Você aprenderá a:
- Criar um checkpointer AgentCoreMemorySaver para persistência automática de estado
- Construir um agente matemático com ferramentas de cálculo
- Persistir estado de conversa automaticamente entre turnos
- Inspecionar histórico de checkpoints e estado do agente
- Gerenciar isolamento de sessão com threads separadas

## Pré-requisitos

- Python 3.10+
- Conta AWS com permissões apropriadas
- Acesso aos modelos do Amazon Bedrock

Vamos começar!

In [ ]:
# Install necessary libraries
!pip install -qr requirements.txt

In [ ]:
# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

In [ ]:
# Import the AgentCoreMemorySaver that we will use as a checkpointer
import os
import logging

from langgraph_checkpoint_aws import AgentCoreMemorySaver
from bedrock_agentcore.memory import MemoryClient

region = os.getenv('AWS_REGION', 'us-west-2')
logging.getLogger("math-agent").setLevel(logging.DEBUG)

# Create or get the memory resource
memory_name = "MathLanggraphAgent"
client = MemoryClient(region_name=region)
memory = client.create_or_get_memory(name=memory_name)
memory_id = memory['id'] # Keep this memory ID for later use

### Configuração de Memória AgentCore

Agora vamos configurar nosso checkpointer de Memória AgentCore que persistirá automaticamente o estado do agente entre turnos.

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

# Initialize checkpointer for state persistence
checkpointer = AgentCoreMemorySaver(memory_id, region_name=region)

# Initialize LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

### Ferramentas Matemáticas

Vamos definir as ferramentas matemáticas que nosso agente usará. Essas ferramentas permitem ao agente realizar cálculos precisos.

In [ ]:
@tool
def add(a: int, b: int):
    """Add two integers and return the result"""
    return a + b


@tool
def multiply(a: int, b: int):
    """Multiply two integers and return the result"""
    return a * b


tools = [add, multiply]

### Implementação do Agente LangGraph

Agora vamos criar nosso agente usando o framework ReAct do LangGraph, integrado com nosso checkpointer AgentCore Memory.

In [ ]:
graph = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful assistant",
    checkpointer=checkpointer,
)

graph

## Passo 4: Executar o Agente LangGraph
Agora podemos executar o agente com nosso sistema de checkpointing de Memória AgentCore integrado.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-1", # REQUIRED: This maps to Bedrock AgentCore session_id under the hood
        "actor_id": "react-agent-1", # REQUIRED: This maps to Bedrock AgentCore actor_id under the hood
    }
}

inputs = {"messages": [{"role": "user", "content": "What is 1337 times 515321? Then add 412 and return the value to me."}]}

#### Parabéns! Seu Agente está pronto!!

### Vamos testar o Agente

Vamos executar uma série de cálculos matemáticos que demonstram como o checkpointer mantém estado entre turnos de conversa.

In [ ]:
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Inspecionando o Estado do Agente

Vamos examinar o estado atual da conversa armazenado no nosso checkpoint de Memória AgentCore.

In [ ]:
for message in graph.get_state(config).values.get("messages"):
    print(f"{message.type}: {message.text()}")
    print("=========================================")

### Visualizando Histórico de Checkpoints

Vamos explorar o histórico de checkpoints para ver como o estado foi preservado em cada turno de conversa.

In [ ]:
for checkpoint in graph.get_state_history(config):
    print(
        f"(Checkpoint ID: {checkpoint.config['configurable']['checkpoint_id']}) # of messages in state: {len(checkpoint.values.get('messages'))}"
    )

### Testando Persistência de Memória

Agora vamos testar o poder do nosso checkpointer criando uma nova instância do agente que pode continuar de onde a conversa anterior parou.

In [ ]:
inputs = {"messages": [{"role": "user", "content": "What were the first calculations I asked you to do?"}]}

for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Iniciando uma Nova Sessão

Vamos demonstrar o isolamento de sessão criando uma nova thread que começa sem contexto de conversa anterior.

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2", # New session ID
        "actor_id": "react-agent-1", # Same Actor ID
    }
}

inputs = {"messages": [{"role": "user", "content": "What values did I ask you to multiply and add?"}]}
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

## Resumo

Neste notebook, demonstramos:

1. Como criar um checkpointer AgentCore Memory para persistência automática de estado
2. Como construir um agente matemático com ferramentas de cálculo integradas
3. Como inspecionar o estado do agente e histórico de checkpoints
4. Como a memória persiste entre instâncias de agentes dentro da mesma sessão
5. Como sessões fornecem isolamento de contexto de conversa

O padrão de checkpointing fornece persistência automática de estado sem código explícito de gerenciamento de memória.

### Limpeza
Vamos deletar a memória para limpar os recursos usados neste notebook.

In [ ]:
#client.delete_memory_and_wait(memory_id = memory_id, max_wait = 300, poll_interval =10)